In [ ]:
# === ContentFactory YouTube VIDEO Worker ===
# Worker: ru.iskhakov2017@gmail.com

from google.colab import drive
drive.mount("/content/drive")

!apt-get update -qq
!apt-get install -y -qq ffmpeg

import os
import subprocess
from pathlib import Path

WORKER_EMAIL = "ru.iskhakov2017@gmail.com"

ROOT = Path("/content/drive/MyDrive/ContentFactory_YouTube")
BOOTSTRAP_PATH = ROOT / "scripts" / "youtube_video_bootstrap_colab.py"
SCRIPT_PATH = ROOT / "scripts" / "youtube_video_worker_colab.py"

print("WORKER_EMAIL:", WORKER_EMAIL)
print("ROOT exists:", ROOT.exists(), ROOT)
print("BOOTSTRAP exists:", BOOTSTRAP_PATH.exists(), BOOTSTRAP_PATH)
print("SCRIPT exists:", SCRIPT_PATH.exists(), SCRIPT_PATH)

if not ROOT.exists():
    raise RuntimeError(
        "ContentFactory_YouTube не найден. "
        "Проверь, что для этого Google-аккаунта создан shortcut в My Drive."
    )

if not BOOTSTRAP_PATH.exists():
    raise RuntimeError(
        "youtube_video_bootstrap_colab.py не найден. "
        "Сначала запусти setup-colab-workers на Windows."
    )

if not SCRIPT_PATH.exists():
    raise RuntimeError(
        "youtube_video_worker_colab.py не найден. "
        "Сначала запусти setup-colab-workers на Windows."
    )

os.environ["CONTENT_FACTORY_WORKER_EMAIL"] = WORKER_EMAIL
os.environ["CONTENT_FACTORY_YOUTUBE_ROOT"] = str(ROOT)
os.environ["CONTENT_FACTORY_VIDEO_QUEUE_MODE"] = "1"
os.environ["CONTENT_FACTORY_MAX_JOBS_PER_RUN"] = "0"
os.environ["CONTENT_FACTORY_POLL_SECONDS"] = "10"
os.environ["CONTENT_FACTORY_IDLE_TIMEOUT_MIN"] = "15"
os.environ["CONTENT_FACTORY_IDLE_EXIT_SECONDS"] = "900"
os.environ["CONTENT_FACTORY_SELF_RECLAIM_STALE_MINUTES"] = "10"
os.environ["CONTENT_FACTORY_SELF_RECLAIM_MAX_ATTEMPTS"] = "3"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["CONTENT_FACTORY_REQUIRE_T4"] = "0"

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
gpu_name = gpu.stdout.strip().splitlines()[0].strip() if gpu.returncode == 0 and gpu.stdout.strip() else ""
print("GPU:", gpu_name or "not available")
if not gpu_name:
    message = "GPU not available. In Colab use Runtime -> Change runtime type -> GPU."
    if os.environ.get("CONTENT_FACTORY_REQUIRE_T4") == "1":
        raise RuntimeError(message)
    print("[WARN]", message)
elif "T4" not in gpu_name.upper():
    message = f"GPU is not T4: {gpu_name}. Continuing because CONTENT_FACTORY_REQUIRE_T4=0."
    if os.environ.get("CONTENT_FACTORY_REQUIRE_T4") == "1":
        raise RuntimeError(message)
    print("[WARN]", message)

%run "/content/drive/MyDrive/ContentFactory_YouTube/scripts/youtube_video_bootstrap_colab.py" --story-slug "Becoming_A_Slut_Wife_Alma" --worker-email "ru.iskhakov2017@gmail.com" --max-jobs-per-run "0" --idle-timeout-min "15" --poll-seconds "10"
